# Jour 2 · Chapitre 3 — Scénario : le marketing veut connaître nos clients
## Formation Beobank · SAS → Python · Corrigé formateur

**L'histoire.** Le service marketing prépare une campagne e-mail pour le printemps. Inès, chargée
de campagne, vous demande :
*"Qui sont nos clients ? Quelle langue, quel âge ? Et surtout : combien ont une adresse e-mail
qu'on peut utiliser ? Je ne connais rien à Python, explique-moi simplement."*

**Comment travailler.** Remplacez chaque `______`. Chaque étape reprend une notion simple de
Pandas, dans l'ordre du cours.

📎 **Matière à relire (même dossier) :** `Pandas.ipynb` (ouvrir, regarder, renommer, choisir,
filtrer, trier, créer des colonnes, texte), `Pandas_dates_et_melange.ipynb` (dates, `.dt`,
`.merge()`) et `Pandas_XOR_explication.ipynb` (le « ou exclusif » `^`).

## Préparation (déjà écrite, exécutez-la)

In [ ]:
# ── Import ────────────────────────────────────────────────────────────────
import pandas as pd
DATA = "../../data/"
print("Prêt.")

---
## Étape 1 — Ouvrir le fichier des clients

In [ ]:
# ── Ouvrir et regarder ────────────────────────────────────────────────────
# Quelle fonction ouvre un fichier CSV ? Quel caractère sépare les colonnes dans nos fichiers ?
tie = pd.read_csv(DATA + "TIE.csv", sep=";", na_values=".")
# Quelle méthode montre les 5 premières lignes ?
tie.head()

---
## Étape 2 — Des noms de colonnes lisibles pour Inès

`COD_LNG_CTR` → `langue`, `DAT_NAI` → `naissance`, `COD_SEX` → `sexe`.

In [ ]:
# ── Renommer et choisir des colonnes ──────────────────────────────────────
# Quelle méthode change le nom des colonnes ?
clients = tie.rename(columns={"COD_LNG_CTR": "langue", "DAT_NAI": "naissance", "COD_SEX": "sexe"})
# Pour garder PLUSIEURS colonnes, on donne une liste entre crochets : complétez la sélection
clients[["NUM_TIE", "langue", "sexe"]].head(3)

---
## Étape 3 — Les clients néerlandophones

*"Combien de clients reçoivent nos e-mails en néerlandais ?"*

In [ ]:
# ── Filtrer ───────────────────────────────────────────────────────────────
# Quel code langue désigne le néerlandais dans le fichier ?
nl = clients[clients["langue"] == "NL"]
# Quelle fonction de base compte les lignes ?
print("Clients néerlandophones :", len(nl))         # → 33

---
## Étape 4 — Calculer l'âge

*"Je veux cibler par tranche d'âge."*

In [ ]:
# ── Dates ─────────────────────────────────────────────────────────────────
# Quelle fonction transforme un texte "1980-01-25" en vraie date ?
clients["naissance"] = pd.to_datetime(clients["naissance"])
# Quel accesseur donne accès aux morceaux d'une date (année, mois, jour) ?
clients["annee"] = clients["naissance"].dt.year
clients["age"] = 2026 - clients["annee"]
# Quelle méthode trie le tableau ? Les plus âgés d'abord : ordre croissant ou non ?
clients.sort_values("age", ascending=False)[["NUM_TIE", "naissance", "age"]].head(3)

---
## Étape 5 — Des tranches d'âge

In [ ]:
# ── Ranger des nombres dans des boîtes ────────────────────────────────────
# Quelle fonction range des nombres dans des intervalles (des « boîtes ») ?
clients["tranche"] = pd.cut(clients["age"], bins=[0, 25, 45, 65, 120],
                              labels=["18-25", "26-45", "46-65", "65+"])
# Quelle méthode compte combien de clients par tranche ?
clients["tranche"].value_counts()

---
## Étape 6 — Les e-mails

*"Les adresses e-mail sont dans un autre fichier. On a quels fournisseurs ?"*

In [ ]:
# ── Jouer avec le texte ───────────────────────────────────────────────────
adr = pd.read_csv(DATA + "TIE_ADR.csv", sep=";", na_values=".")
# Quelle méthode texte coupe "BART@GMAIL.COM" en deux morceaux autour du "@" ?
# Après le découpage, quel accesseur permet de prendre le 2e morceau de chaque ligne ?
domaines = adr["ADR_EMA"].str.split("@").str[1]
# Quelle méthode texte met tout en minuscules ?
domaines.str.lower().value_counts().head(3)

---
## Étape 7 — Coller les deux tableaux

*"Mets l'e-mail à côté de chaque client."*

In [ ]:
# ── Fusionner ─────────────────────────────────────────────────────────────
# Quelle méthode colle deux tableaux grâce à une colonne commune ?
# Quelle colonne existe dans les DEUX tableaux ? On garde tous les clients même sans e-mail : quel type de jointure ?
campagne = clients.merge(adr[["NUM_TIE", "ADR_EMA"]], on="NUM_TIE", how="left")
# Quelle méthode renvoie True quand une case N'EST PAS vide ?
print("Clients avec e-mail :", campagne["ADR_EMA"].notna().sum(), "sur", len(campagne))   # → 83 sur 100

---
## 🏁 Étape finale — La cible de la campagne

*"Je veux SOIT les francophones, SOIT les moins de 30 ans — mais pas les deux à la fois : les
jeunes francophones ont déjà reçu une autre campagne."*

In [ ]:
# ── Le OU exclusif (XOR) ──────────────────────────────────────────────────
est_fr = campagne["langue"] == "FR"
est_jeune = campagne["age"] < 30
# Quel opérateur signifie « l'un OU l'autre, mais PAS les deux » ?
cible = campagne[(est_fr ^ est_jeune) & campagne["ADR_EMA"].notna()]
print("Taille de la cible :", len(cible))
cible[["NUM_TIE", "langue", "age", "ADR_EMA"]].head()

✍️ **Exemple de réponse :** *"La cible compte les clients francophones de 30 ans et plus et les
moins de 30 ans néerlandophones, uniquement s'ils ont un e-mail. Les jeunes francophones sont
exclus volontairement (autre campagne). 17 clients sur 100 n'ont pas d'e-mail : il faudra les
contacter par courrier."*